In [84]:
import torch

x = torch.tensor(5.0, requires_grad=True)
f = x ** 2
f

tensor(25., grad_fn=<PowBackward0>)

In [85]:
f.backward()

In [86]:
x.grad

tensor(10.)

In [87]:
learning_rate = 0.1
with torch.no_grad():
    x -= learning_rate * x.grad

In [88]:
learning_rate = 0.1
x = torch.tensor(5.0, requires_grad=True)
for iteration in range(100):
    f = x ** 2
    f.backward()
    with torch.no_grad():
        x -= learning_rate * x.grad

    x.grad.zero_()

### Real Example

In [89]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

housing = fetch_california_housing()
housing

{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
           37.88      , -122.23      ],
        [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
           37.86      , -122.22      ],
        [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
           37.85      , -122.24      ],
        ...,
        [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
           39.43      , -121.22      ],
        [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
           39.43      , -121.32      ],
        [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
           39.37      , -121.24      ]], shape=(20640, 8)),
 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894], shape=(20640,)),
 'frame': None,
 'target_names': ['MedHouseVal'],
 'feature_names': ['MedInc',
  'HouseAge',
  'AveRooms',
  'AveBedrms',
  'Population',
  'AveOccup',
  'Latitude',
  'Longitude'],
 'DESCR': 

In [90]:
X_train_full, X_test, y_train_full, y_test = train_test_split(
    housing.data, housing.target, random_state=123
)
print(X_train_full.shape, X_test.shape)
print(f'X_train_full is {(X_train_full.shape[0] / (X_train_full.shape[0] + X_test.shape[0]) * 100)}%\
      X_test is {(X_test.shape[0] / (X_train_full.shape[0] + X_test.shape[0]) * 100)}%')

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, random_state=123
)
print(X_train.shape, X_valid.shape)
print(f'X_train is {X_train.shape[0] / (X_train.shape[0] + X_valid.shape[0]) * 100}%\
      X_valid is {X_valid.shape[0] / (X_train.shape[0] + X_valid.shape[0]) * 100}%')

(15480, 8) (5160, 8)
X_train_full is 75.0%      X_test is 25.0%
(11610, 8) (3870, 8)
X_train is 75.0%      X_valid is 25.0%


In [91]:
X_train

array([[   3.3804    ,   38.        ,    5.83009709, ...,    3.55825243,
          33.9       , -118.25      ],
       [   1.398     ,   14.        ,    5.65422397, ...,    1.98624754,
          37.51      , -119.99      ],
       [   6.9931    ,   25.        ,   10.17318436, ...,    1.94972067,
          38.36      , -122.26      ],
       ...,
       [  15.0001    ,   52.        ,    8.78061224, ...,    3.51020408,
          34.08      , -118.34      ],
       [   3.125     ,    9.        ,    5.55409219, ...,    3.28222013,
          34.54      , -117.32      ],
       [   2.1935    ,   31.        ,    4.79402985, ...,    3.13134328,
          34.16      , -117.34      ]], shape=(11610, 8))

In [92]:
X_train = torch.FloatTensor(X_train)
X_train

tensor([[   3.3804,   38.0000,    5.8301,  ...,    3.5583,   33.9000,
         -118.2500],
        [   1.3980,   14.0000,    5.6542,  ...,    1.9862,   37.5100,
         -119.9900],
        [   6.9931,   25.0000,   10.1732,  ...,    1.9497,   38.3600,
         -122.2600],
        ...,
        [  15.0001,   52.0000,    8.7806,  ...,    3.5102,   34.0800,
         -118.3400],
        [   3.1250,    9.0000,    5.5541,  ...,    3.2822,   34.5400,
         -117.3200],
        [   2.1935,   31.0000,    4.7940,  ...,    3.1313,   34.1600,
         -117.3400]])

In [93]:
X_valid = torch.FloatTensor(X_valid)
X_test = torch.FloatTensor(X_test)

In [94]:
means = X_train.mean(dim=0, keepdim=True)

In [95]:
means

tensor([[ 3.8756e+00,  2.8531e+01,  5.4368e+00,  1.0979e+00,  1.4326e+03,
          3.1159e+00,  3.5631e+01, -1.1957e+02]])

In [96]:
stds = X_train.std(dim=0, keepdim=True)

In [97]:
stds

tensor([[1.9053e+00, 1.2533e+01, 2.6640e+00, 5.2507e-01, 1.1161e+03, 1.2652e+01,
         2.1311e+00, 2.0035e+00]])

In [98]:
X_train = (X_train - means) / stds
X_valid = (X_valid - means) / stds
X_test = (X_test - means) / stds

##### PyTorch expects the targets to have one row per sample, so let's reshape the targets to be column vectors:

In [99]:
y_train = torch.FloatTensor(y_train).view(-1, 1)
y_valid = torch.FloatTensor(y_valid).view(-1, 1)
y_test = torch.FloatTensor(y_test).view(-1, 1)

In [100]:
torch.manual_seed(123)
n_features = X_train.shape[1]
w = torch.rand((n_features, 1), requires_grad=True)
b = torch.tensor(0., requires_grad=True)

In [101]:
learning_rate = 0.4
n_epochs = 20
for epoch in range(n_epochs):
    y_pred = X_train @ w + b
    loss = (y_pred - y_train).pow(2).mean()
    loss.backward()
    with torch.no_grad():
        b -= learning_rate * b.grad
        w -= learning_rate * w.grad
        b.grad.zero_()
        w.grad.zero_()
    print(f'Epoch {epoch + 1}/{n_epochs}, Loss: {loss.item()}')

Epoch 1/20, Loss: 7.004919052124023
Epoch 2/20, Loss: 1.0551552772521973
Epoch 3/20, Loss: 0.7151469588279724
Epoch 4/20, Loss: 0.6581979990005493
Epoch 5/20, Loss: 0.634869396686554
Epoch 6/20, Loss: 0.6212378740310669
Epoch 7/20, Loss: 0.6115010380744934
Epoch 8/20, Loss: 0.603682279586792
Epoch 9/20, Loss: 0.5970300436019897
Epoch 10/20, Loss: 0.5912222266197205
Epoch 11/20, Loss: 0.5860950350761414
Epoch 12/20, Loss: 0.5815466642379761
Epoch 13/20, Loss: 0.5775026679039001
Epoch 14/20, Loss: 0.5739028453826904
Epoch 15/20, Loss: 0.5706958770751953
Epoch 16/20, Loss: 0.5678371787071228
Epoch 17/20, Loss: 0.5652875900268555
Epoch 18/20, Loss: 0.5630125999450684
Epoch 19/20, Loss: 0.5609814524650574
Epoch 20/20, Loss: 0.5591670870780945


<i style="font-family: 'italic'; color: #27b034; font-size: 1.3em;">
PyTorch tracks all operations to compute gradients. If we update parameters without <b style='font-size: 0.8em; color: red'>no_grad()</b>, PyTorch will also track those updates, which is wrong because updates should not be part of gradient computation. So we use <b style='font-size: 0.8em; color: red'>no_grad()</b>to prevent tracking parameter updates.
</i>

In [102]:
X_new = X_test[:3]
with torch.no_grad():
    y_pred = X_new @ w + b

In [103]:
y_pred

tensor([[2.3276],
        [1.5378],
        [2.0075]])